In [35]:
from typing import (
    TypedDict,
    Literal,
    Annotated,
)

import operator
import os
from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage,
    AIMessage,
)
from langchain_core.tools import tool
# from langchain.agents import create_agent

from langgraph.graph import (
    StateGraph,
    START,
    END,
    MessagesState,
)
from langgraph.graph.message import add_messages
from langgraph.types import (
    Command,
    Send,
)
from langgraph.checkpoint.memory import InMemorySaver

In [36]:
load_dotenv()

True

In [37]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found - add it to your .env file.")
    
model = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    max_retries=3,   # free models hit 429 rate limits often
    timeout=120,
)

In [38]:
# agent state

class AgentState(TypedDict):
    Task: str
    messages: Annotated[list, add_messages]
    output: str
    route: str

In [39]:
# creating Specialist agents for Frontend teams

# react agent
def react_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a React specialist. Handle all React component and frontend tasks.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

# ui agent
def ui_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a UI specialist. Build clean, accessible user interfaces and handle all UI releated task.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

# Accessibility Agent
def accessibility_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an Accessibility specialist. Ensure all UI components meet WCAG 2.1 AA standards: semantic HTML, keyboard navigation, screen reader support, color contrast, and proper ARIA usage.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

In [40]:
# creating Specialist agents for backend teams

# api agent
def api_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an API specialist. Design RESTful APIs, handle routing, validation, serialization, and ensure consistent error responses.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

# auth agent
def auth_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an Authentication specialist. Implement secure auth: JWT/OAuth2, token refresh, password hashing, MFA, session management, and RBAC.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

# microservice agent
def microservice_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are a Microservice specialist. Design and review microservice architectures, service boundaries, inter-service communication, resilience, and deployment patterns based on the requirements provided.")
    response = model.invoke([system] + state["messages"])
    return {"messages": [response], "output": response.content}

In [41]:
# creating Specialist agents for database teams

# database agents
def sql_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert SQL specialist. Help with database and SQL tasks by producing accurate, efficient, and secure queries. Use only the schema and requirements provided; if essential details are missing, ask a concise clarifying question. Briefly explain assumptions when needed, and return only the requested result.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}

def schema_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert database schema designer. Help create, review, and improve database schemas based on the requirements provided. Recommend appropriate tables, columns, data types, primary and foreign keys, constraints, and relationships. Prioritize data integrity, clarity, and maintainability. Ask a concise clarifying question if essential requirements are missing, and return only the requested result.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}

def Query_optimization_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert SQL query optimization specialist. Analyze SQL queries and the provided schema to identify ways to improve performance without changing the results. Consider indexes, joins, filtering, query plans, and database-specific behavior. State any assumptions, ask a concise clarifying question if essential details are missing, and return only the requested result.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}

In [42]:
# creating Specialist agents for QA teams

# unit test Agent
def Unit_test_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert unit testing specialist. Help write focused tests for individual functions, methods, and components. Cover expected behavior, edge cases, and error conditions using the project’s existing testing framework and conventions. Keep tests deterministic and independent. If requirements or code are missing, ask a concise clarifying question; otherwise return only the requested tests or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}
# Integration test Agent
def integration_test_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert integration testing specialist. Help test how multiple components, services, or modules work together. Follow the project’s existing testing framework and conventions, and cover key data flows, boundaries, and failure cases. Keep tests reliable and avoid unnecessary external dependencies. If requirements or code are missing, ask a concise clarifying question; otherwise return only the requested tests or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}
# regresion Agent
def regression_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert regression testing specialist. Help create tests that catch previously fixed bugs and prevent existing behavior from breaking. Use the provided bug details, expected behavior, and project testing conventions to write focused, repeatable tests. If essential context is missing, ask a concise clarifying question; otherwise return only the requested tests or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}

In [43]:
# creating Specialist agents for DevOps teams

# Docker Agent
def docker_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert Docker specialist. Help create, review, and troubleshoot Dockerfiles, images, and Compose configurations. Recommend secure, efficient, and reproducible container practices, following the project’s requirements and conventions. If essential details are missing, ask a concise clarifying question; otherwise return only the requested configuration or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}
# Kubernetes Agent
def kubernetes_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert Kubernetes specialist. Help design, write, review, and troubleshoot Kubernetes resources such as Deployments, Services, ConfigMaps, and Ingresses. Prioritize reliability, security, and maintainability, and use the requirements and environment details provided. If essential information is missing, ask a concise clarifying question; otherwise return only the requested manifests or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}
# CI/CD Agent
def cicd_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert CI/CD specialist. Help design, write, and troubleshoot continuous integration and deployment pipelines using the platform and project requirements provided. Focus on reliable automation, clear validation steps, secure handling of secrets, and safe deployments. If essential details are missing, ask a concise clarifying question; otherwise return only the requested pipeline configuration or guidance.")
    response = model.invoke([system] + state["messages"])
    return {"message": [response], "output": response.content}

In [44]:
# creating Specialist agents for security Team

# Vulnerability Agent
def vulnerability_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert application security specialist. Review the provided code or configuration for potential vulnerabilities. For each finding, describe the affected area, likely impact, severity, and a safe remediation. Focus on defensive analysis and do not expose secrets or provide instructions for exploiting systems. If essential context is missing, ask a concise clarifying question; otherwise return only the requested findings or guidance.")
    response = model.invoke([system] + state['messages'])
    return {'messages': [response], 'output': response.content}
# IAM Agent
def iam_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert Identity and Access Management (IAM) specialist. Help design, review, and troubleshoot identities, roles, permissions, and access policies. Apply least privilege, separation of duties, and secure authentication practices. Explain risks and suggest practical improvements based on the requirements provided. If essential details are missing, ask a concise clarifying question; otherwise return only the requested policy or guidance.")
    response = model.invoke([system] + state["messages"])
    return {'messages': [response], 'output': response.content}
# Secrets Agent
def secrets_agent(state: AgentState) -> AgentState:
    system = SystemMessage(content="You are an expert secrets-management specialist. Help identify and prevent unsafe handling of credentials, tokens, keys, and other sensitive values. Recommend secure storage, access controls, rotation, and safe injection into applications and pipelines. Never repeat or expose secret values; redact them if they appear in the provided material. If essential details are missing, ask a concise clarifying question; otherwise return only the requested findings or guidance.")
    response = model.invoke([system] + state["messages"])
    return {'messages': [response], 'output': response.content}

In [45]:
# frontend-team supervisor
def frontend_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the Frontend Team Supervisor. Coordinate frontend work across these roles:
- Frontend Developer: components, application logic, and styling
- UI/UX Specialist: usability, layout, and interaction design
- Accessibility Specialist: keyboard access, semantics, and screen-reader support
- Frontend Test Specialist: component and interaction tests

Analyze the request, identify which roles are relevant, and provide one clear,
integrated response with actionable steps. Do not claim that work was delegated,
implemented, or tested unless that actually happened.
""")
    response = model.invoke([system] + state["messages"])
    return {
        'messages': [response],
        'output': response.content
    }

In [46]:
# backend team supervisor
def backend_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the Backend Team Supervisor. Coordinate backend work across these roles:
- API Specialist: endpoint design, contracts, and request/response handling
- Backend Developer: server-side application logic and integrations
- Performance Specialist: scalability, latency, and resource efficiency
- Reliability Specialist: error handling, resilience, and observability

Analyze the request, identify which roles are relevant, and provide one clear,
integrated response with actionable steps. Consider maintainability, security,
and compatibility with the existing system. Do not claim that work was delegated,
implemented, or tested unless that actually happened.
""")
    response = model.invoke([system] + state["messages"])
    return {
        'messages': [response],
        'output': response.content
    }

In [47]:
# database team supervisor
def database_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the Database Team Supervisor. Coordinate database work across these roles:
- Schema Agent: tables, relationships, data types, and constraints
- SQL Agent: accurate, secure queries and database operations
- Query Optimization Agent: indexes, execution plans, and query performance

Analyze the request, identify which roles are relevant, and provide one clear,
integrated response with actionable steps. Prioritize data integrity, security,
and performance, and account for the database engine and schema provided. Do not
claim that work was delegated, implemented, or tested unless that actually happened.
""")
    response = model.invoke([system] + state["messages"])
    return {
        'messages': [response],
        'output': response.content
    }

In [48]:
# QA team supervisor
def qa_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the QA Team Supervisor. Coordinate quality assurance work across these roles:
- Unit Test Agent: tests for individual functions, methods, and components
- Integration Test Agent: tests for interactions between modules and services
- Regression Test Agent: tests that prevent previously fixed bugs from returning

Analyze the request, identify the relevant test coverage, and provide one clear,
integrated response with actionable steps. Follow the project’s existing test
framework and conventions. Prioritize reliable tests for expected behavior, edge
cases, and failures. Do not claim that work was delegated, implemented, or tested
unless that actually happened.
""")
    response = model.invoke([system] + state["messages"])
    return {
        'messages': [response],
        'output': response.content
    }

In [49]:
# DevOps team supervisor
def devops_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the DevOps Team Supervisor. Coordinate DevOps work across these roles:
- Docker Agent: container images and Docker Compose configurations
- Kubernetes Agent: deployments, services, and cluster resources
- CI/CD Agent: build, test, and deployment pipelines

Analyze the request, identify which roles are relevant, and provide one clear,
integrated response with actionable steps. Prioritize secure, reliable, and
reproducible builds, deployments, and operations. Do not claim that work was
delegated, implemented, or tested unless that actually happened.
""")
    response = model.invoke([system] + state["messages"])
    return {
        'messages': [response],
        'output': response.content
    }

In [50]:
# security team supervisor
def security_team_supervisor(state: AgentState) -> AgentState:
    system = SystemMessage(content="""
You are the Security Team Supervisor. Coordinate security work across these roles:
- Vulnerability Agent: defensive review of code and configuration for security risks
- IAM Agent: identities, authentication, roles, and access policies
- Secrets Agent: safe storage, access, rotation, and handling of sensitive values

Analyze the request, identify the relevant security areas, and provide one clear,
integrated response with actionable, prioritized recommendations. Apply least
privilege and never expose secret values. Do not provide instructions to exploit
systems or claim that work was delegated, implemented, or tested unless that
actually happened.
""")
    response = model.invoke([system] + state["messages"])
    return {
        'messages': [response],
        'output': response.content
    }

In [51]:
# frontend_team graph
frontend_team_subgraph = StateGraph(AgentState)
# nodes
frontend_team_subgraph.add_node("frontend_team", frontend_team_supervisor)
frontend_team_subgraph.add_node("react_agent", react_agent)
frontend_team_subgraph.add_node("ui_agent", ui_agent)
frontend_team_subgraph.add_node("accessibility_agent", accessibility_agent)
# edges
frontend_team_subgraph.add_edge(START, "frontend_team")
frontend_team_subgraph.add_edge("frontend_team", "react_agent")
frontend_team_subgraph.add_edge("react_agent", "frontend_team")
frontend_team_subgraph.add_edge("frontend_team", "ui_agent")
frontend_team_subgraph.add_edge("ui_agent", "frontend_team")
frontend_team_subgraph.add_edge("frontend_team", "accessibility_agent")
frontend_team_subgraph.add_edge("accessibility_agent", "frontend_team")
fronteam_team = frontend_team_subgraph.compile()

## Notes: fixes applied

- Standardized the conversation state field to `messages` and used it consistently in the state definition, specialist agents, and team supervisors.
- Standardized agent return values to use `messages: [response]` so they work with the `add_messages` reducer.
- Renamed `microsoft_agent` to `microservice_agent` and replaced its system prompt with a Microservice specialist prompt.
- Fixed `vulnerability_agent` to use the correctly spelled `response` variable and read/write the plural `messages` field.
- Changed the frontend subgraph schema from `StateGraph(START)` to `StateGraph(AgentState)`; `START` remains the graph's start node marker.
